# 05 — XGBoost Model

This notebook trains a nonlinear XGBoost model to predict first-attempt correctness
using the same chronological train, validation, and test files as notebook 04. Keeping
the evaluation contract identical makes comparison with the logistic-regression
baseline fair and auditable.

The workflow uses validation-only model selection and threshold selection. The held-out
test set is evaluated only after all choices are fixed.


## Modeling and evaluation protocol

- Select predictors from the feature dictionary.
- Prefer raw `opportunity` for the tree model and remove `log1p_opportunity`.
- Fit three compact, purposeful hyperparameter candidates with early stopping on the
  validation set.
- Select the candidate with the lowest validation log loss.
- Fix the selected number of boosting rounds and refit on train + validation without
  examining test performance.
- Compare probability quality, discrimination, and threshold-dependent results against
  the prevalence-only baseline and notebook 04's logistic-regression results.
- Use native XGBoost gain and SHAP contributions for global and local explanations.

Explanations describe model behavior and conditional associations. They are not causal
claims or measures of a student's fixed ability.


## Imports and notebook setup


In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xgboost as xgb
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.special import expit
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings("default")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

RANDOM_STATE = 42

print(f"Python:   {sys.version.split()[0]}")
print(f"pandas:   {pd.__version__}")
print(f"NumPy:    {np.__version__}")
print(f"XGBoost:  {xgb.__version__}")


Python:   3.13.14
pandas:   3.0.5
NumPy:    2.5.2
XGBoost:  3.4.1


## Project paths


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    '''Find the nearest parent directory containing pyproject.toml.'''
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml.")


PROJECT_ROOT = find_project_root()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "data_dictionary"
    / "skill_builder_data_feature_eng_data_dictionary.csv"
)
SPLIT_DATA_PATHS = {
    split_name: PROCESSED_DATA_DIR / f"skill_builder_data_feature_eng_{split_name}.csv"
    for split_name in ("train", "validation", "test")
}

for required_path in [FEATURE_DICTIONARY_PATH, *SPLIT_DATA_PATHS.values()]:
    assert required_path.is_file(), required_path

print(f"Project root: {PROJECT_ROOT}")
for split_name, split_path in SPLIT_DATA_PATHS.items():
    print(f"{split_name.title()}: {split_path.relative_to(PROJECT_ROOT)}")


Project root: W:\Workstation ExtDrive\007 Data Science\003 Data Science Projects\2026_p019 assistments_2009_2010
Train: data\processed\skill_builder_data_feature_eng_train.csv
Validation: data\processed\skill_builder_data_feature_eng_validation.csv
Test: data\processed\skill_builder_data_feature_eng_test.csv


## Feature policy from the data dictionary


In [3]:
feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH, keep_default_na=False)
assert feature_dictionary["column_name"].is_unique

include_mask = feature_dictionary["model_inclusion"].str.startswith("Include")
candidate_rows = feature_dictionary.loc[include_mask].copy()

# Tree models can learn a nonlinear opportunity relationship directly.
candidate_rows = candidate_rows.loc[
    candidate_rows["column_name"].ne("log1p_opportunity")
]
feature_names = candidate_rows["column_name"].tolist()

binary_mask = (
    candidate_rows["data_type"].str.contains("binary", case=False, na=False)
    | candidate_rows["model_inclusion"].str.contains("one-hot", case=False, na=False)
)
binary_features = candidate_rows.loc[binary_mask, "column_name"].tolist()
numeric_features = candidate_rows.loc[~binary_mask, "column_name"].tolist()

for forbidden_column in {
    "user_id",
    "order_id",
    "correct",
    "data_split",
    "attempt_count",
    "hint_count",
    "bottom_hint",
}:
    assert forbidden_column not in feature_names, forbidden_column

assert "opportunity" in feature_names
assert "log1p_opportunity" not in feature_names
assert set(feature_names) == set(binary_features) | set(numeric_features)

feature_policy_summary = pd.DataFrame(
    {
        "feature_block": ["Numeric/count", "Binary/one-hot", "Total"],
        "feature_count": [len(numeric_features), len(binary_features), len(feature_names)],
        "tree_handling": [
            "Native numeric split",
            "Native 0/1 split",
            "No scaling; missing values supported natively",
        ],
    }
)
display(feature_policy_summary)


,feature_block,feature_count,tree_handling
0,Numeric/count,38,Native numeric split
1,Binary/one-hot,132,Native 0/1 split
2,Total,170,No scaling; missing values supported natively


## Load chronological train, validation, and test data


In [4]:
TARGET = "correct"
METADATA_COLUMNS = ["user_id", "order_id", "data_split"]
use_columns = [*METADATA_COLUMNS, TARGET, *feature_names]

dtype_map = {
    **{column: "int8" for column in binary_features},
    **{column: "float32" for column in numeric_features},
    TARGET: "int8",
    "user_id": "int64",
    "order_id": "int64",
}

split_frames = {
    split_name: pd.read_csv(
        split_path,
        usecols=use_columns,
        dtype=dtype_map,
        low_memory=False,
    )
    for split_name, split_path in SPLIT_DATA_PATHS.items()
}
train = split_frames["train"]
validation = split_frames["validation"]
test = split_frames["test"]

for split_name, frame in split_frames.items():
    assert len(frame) > 0
    assert frame["data_split"].eq(split_name).all()
    assert set(frame[TARGET].unique()).issubset({0, 1})
    assert frame[feature_names].columns.tolist() == feature_names
    assert not np.isinf(frame[numeric_features].to_numpy()).any()

assert train["order_id"].max() < validation["order_id"].min()
assert validation["order_id"].max() < test["order_id"].min()

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(frame),
            "students": frame["user_id"].nunique(),
            "order_id_min": frame["order_id"].min(),
            "order_id_max": frame["order_id"].max(),
            "correct_rate": frame[TARGET].mean(),
            "missing_predictor_cells": int(frame[feature_names].isna().sum().sum()),
        }
        for split_name, frame in split_frames.items()
    ]
)
display(split_summary)


,split,rows,students,order_id_min,order_id_max,correct_rate,missing_predictor_cells
0,train,181570,3005,20224180,33447607,0.6559,0
1,validation,38907,1789,33447620,38144345,0.6092,0
2,test,38909,87,38144353,38310202,0.7169,0


## Evaluation helpers


In [5]:
def classification_metrics(
    y_true: pd.Series | np.ndarray,
    probability: np.ndarray,
    threshold: float,
    model_name: str,
    split_name: str,
) -> dict[str, float | str]:
    '''Return discrimination, calibration, and threshold-dependent metrics.'''
    prediction = (probability >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "split": split_name,
        "model": model_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "log_loss": log_loss(y_true, probability, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision_correct": precision_score(y_true, prediction, zero_division=0),
        "recall_correct": recall_score(y_true, prediction, zero_division=0),
        "specificity_incorrect": tn / (tn + fp),
        "f1_correct": f1_score(y_true, prediction, zero_division=0),
    }


def build_xgboost_model(
    model_parameters: dict[str, float | int],
    n_estimators: int = 2_000,
    early_stopping_rounds: int | None = 75,
) -> XGBClassifier:
    '''Create a reproducible CPU histogram XGBoost classifier.'''
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_estimators=n_estimators,
        early_stopping_rounds=early_stopping_rounds,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        **model_parameters,
    )


## Tune compact XGBoost candidates on validation log loss

The candidates vary tree depth, learning rate, sampling, and regularization without an
expansive grid search. Each uses early stopping, so the validation set selects both the
configuration and effective number of boosting rounds.


In [6]:
X_train = train[feature_names]
y_train = train[TARGET]
X_validation = validation[feature_names]
y_validation = validation[TARGET]

CANDIDATE_CONFIGURATIONS = [
    {
        "configuration": "shallow_conservative",
        "max_depth": 3,
        "learning_rate": 0.05,
        "min_child_weight": 10,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
        "reg_alpha": 0.0,
    },
    {
        "configuration": "balanced",
        "max_depth": 5,
        "learning_rate": 0.05,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 3.0,
        "reg_alpha": 0.0,
    },
    {
        "configuration": "deeper_regularized",
        "max_depth": 7,
        "learning_rate": 0.03,
        "min_child_weight": 10,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 5.0,
        "reg_alpha": 0.10,
    },
]

tuning_records = []
candidate_models = {}
for configuration in CANDIDATE_CONFIGURATIONS:
    configuration_name = configuration["configuration"]
    model_parameters = {
        key: value for key, value in configuration.items() if key != "configuration"
    }
    candidate_model = build_xgboost_model(model_parameters)
    fit_started = time.perf_counter()
    candidate_model.fit(
        X_train,
        y_train,
        eval_set=[(X_validation, y_validation)],
        verbose=False,
    )
    validation_probability = candidate_model.predict_proba(X_validation)[:, 1]
    tuning_record = {
        "configuration": configuration_name,
        **model_parameters,
        "best_boosting_rounds": int(candidate_model.best_iteration + 1),
        "validation_log_loss": log_loss(
            y_validation, validation_probability, labels=[0, 1]
        ),
        "validation_roc_auc": roc_auc_score(y_validation, validation_probability),
        "validation_average_precision": average_precision_score(
            y_validation, validation_probability
        ),
        "validation_brier_score": brier_score_loss(
            y_validation, validation_probability
        ),
        "fit_seconds": time.perf_counter() - fit_started,
    }
    tuning_records.append(tuning_record)
    candidate_models[configuration_name] = candidate_model

tuning_results = pd.DataFrame(tuning_records).sort_values(
    ["validation_log_loss", "configuration"], ignore_index=True
)
BEST_CONFIGURATION_NAME = str(tuning_results.loc[0, "configuration"])
BEST_BOOSTING_ROUNDS = int(tuning_results.loc[0, "best_boosting_rounds"])
BEST_PARAMETERS = {
    key: value
    for key, value in next(
        item
        for item in CANDIDATE_CONFIGURATIONS
        if item["configuration"] == BEST_CONFIGURATION_NAME
    ).items()
    if key != "configuration"
}
selection_model = candidate_models[BEST_CONFIGURATION_NAME]

display(tuning_results.style.format(precision=4))
print(f"Selected configuration: {BEST_CONFIGURATION_NAME}")
print(f"Selected boosting rounds: {BEST_BOOSTING_ROUNDS:,}")


,configuration,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,best_boosting_rounds,validation_log_loss,validation_roc_auc,validation_average_precision,validation_brier_score,fit_seconds
0,balanced,5,0.0500,5,0.8500,0.8500,3.0000,0.0000,885,0.5285,0.7861,0.8290,0.1766,12.4408
1,deeper_regularized,7,0.0300,10,0.8500,0.8500,5.0000,0.1000,936,0.5286,0.7854,0.8292,0.1767,15.8450
2,shallow_conservative,3,0.0500,10,0.8000,0.8000,5.0000,0.0000,1525,0.5306,0.7835,0.8261,0.1774,19.3832


Selected configuration: balanced
Selected boosting rounds: 885


## Choose the operating threshold on validation data


In [7]:
validation_probability = selection_model.predict_proba(X_validation)[:, 1]
threshold_records = []
for threshold in np.linspace(0.10, 0.90, 161):
    validation_prediction = (validation_probability >= threshold).astype("int8")
    threshold_records.append(
        {
            "threshold": threshold,
            "balanced_accuracy": balanced_accuracy_score(
                y_validation, validation_prediction
            ),
            "accuracy": accuracy_score(y_validation, validation_prediction),
            "precision_correct": precision_score(
                y_validation, validation_prediction, zero_division=0
            ),
            "recall_correct": recall_score(
                y_validation, validation_prediction, zero_division=0
            ),
            "f1_correct": f1_score(
                y_validation, validation_prediction, zero_division=0
            ),
        }
    )

threshold_results = pd.DataFrame(threshold_records)
threshold_results["distance_from_0_5"] = (
    threshold_results["threshold"] - 0.5
).abs()
threshold_results = threshold_results.sort_values(
    ["balanced_accuracy", "distance_from_0_5"],
    ascending=[False, True],
    ignore_index=True,
)
SELECTED_THRESHOLD = float(threshold_results.loc[0, "threshold"])

validation_prevalence_probability = np.full(len(validation), y_train.mean())
validation_performance = pd.DataFrame(
    [
        classification_metrics(
            y_validation, validation_probability, 0.5, "XGBoost", "validation"
        ),
        classification_metrics(
            y_validation,
            validation_probability,
            SELECTED_THRESHOLD,
            "XGBoost (selected threshold)",
            "validation",
        ),
        classification_metrics(
            y_validation,
            validation_prevalence_probability,
            0.5,
            "Training-prevalence baseline",
            "validation",
        ),
    ]
)

print(f"Validation-selected threshold: {SELECTED_THRESHOLD:.3f}")
display(threshold_results.head(10).drop(columns="distance_from_0_5"))
display(validation_performance.style.format(precision=4))


Validation-selected threshold: 0.635


,threshold,balanced_accuracy,accuracy,precision_correct,recall_correct,f1_correct
0,0.6350,0.7117,0.7160,0.7873,0.7313,0.7583
1,0.6400,0.7110,0.7138,0.7890,0.7237,0.7549
2,0.6300,0.7107,0.7168,0.7841,0.7384,0.7606
3,0.5950,0.7103,0.7268,0.7703,0.7859,0.7780
4,0.6450,0.7102,0.7116,0.7905,0.7164,0.7517
5,0.6050,0.7101,0.7242,0.7732,0.7744,0.7738
6,0.6150,0.7101,0.7212,0.7768,0.7608,0.7688
7,0.6100,0.7100,0.7226,0.7749,0.7676,0.7712
8,0.6200,0.7099,0.7194,0.7788,0.7534,0.7659
9,0.6250,0.7098,0.7177,0.7809,0.7459,0.7630


,split,model,threshold,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision_correct,recall_correct,specificity_incorrect,f1_correct
0,validation,XGBoost,0.5000,0.7861,0.8290,0.5285,0.1766,0.7386,0.6986,0.7394,0.8815,0.5158,0.8042
1,validation,XGBoost (selected threshold),0.6350,0.7861,0.8290,0.5285,0.1766,0.7160,0.7117,0.7873,0.7313,0.6920,0.7583
2,validation,Training-prevalence baseline,0.5000,0.5000,0.6092,0.6738,0.2403,0.6092,0.5000,0.6092,1.0000,0.0000,0.7571


## Refit on train + validation and evaluate the held-out test set


In [8]:
development = pd.concat([train, validation], ignore_index=True)
X_development = development[feature_names]
y_development = development[TARGET]
X_test = test[feature_names]
y_test = test[TARGET]

final_model = build_xgboost_model(
    BEST_PARAMETERS,
    n_estimators=BEST_BOOSTING_ROUNDS,
    early_stopping_rounds=None,
)
final_fit_started = time.perf_counter()
final_model.fit(X_development, y_development, verbose=False)
test_probability = final_model.predict_proba(X_test)[:, 1]
final_fit_seconds = time.perf_counter() - final_fit_started

test_prevalence_probability = np.full(len(test), y_development.mean())
test_performance = pd.DataFrame(
    [
        classification_metrics(y_test, test_probability, 0.5, "XGBoost", "test"),
        classification_metrics(
            y_test,
            test_probability,
            SELECTED_THRESHOLD,
            "XGBoost (validation-selected threshold)",
            "test",
        ),
        classification_metrics(
            y_test,
            test_prevalence_probability,
            0.5,
            "Development-prevalence baseline",
            "test",
        ),
    ]
)
performance_table = pd.concat(
    [validation_performance, test_performance], ignore_index=True
)

print(f"Final refit time: {final_fit_seconds:,.2f} seconds")
display(performance_table.style.format(precision=4))


Final refit time: 9.98 seconds


,split,model,threshold,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision_correct,recall_correct,specificity_incorrect,f1_correct
0,validation,XGBoost,0.5000,0.7861,0.8290,0.5285,0.1766,0.7386,0.6986,0.7394,0.8815,0.5158,0.8042
1,validation,XGBoost (selected threshold),0.6350,0.7861,0.8290,0.5285,0.1766,0.7160,0.7117,0.7873,0.7313,0.6920,0.7583
2,validation,Training-prevalence baseline,0.5000,0.5000,0.6092,0.6738,0.2403,0.6092,0.5000,0.6092,1.0000,0.0000,0.7571
3,test,XGBoost,0.5000,0.7000,0.8454,0.5443,0.1823,0.7328,0.5919,0.7600,0.9168,0.2671,0.8311
4,test,XGBoost (validation-selected threshold),0.6350,0.7000,0.8454,0.5443,0.1823,0.6974,0.6317,0.7924,0.7831,0.4804,0.7877
5,test,Development-prevalence baseline,0.5000,0.5000,0.7169,0.6068,0.2078,0.7169,0.5000,0.7169,1.0000,0.0000,0.8351


## Performance curves and probability diagnostics


In [9]:
diagnostic_figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "ROC curves",
        "Precision-recall curves",
        "Calibration curves",
        "Test predicted probabilities by outcome",
    ),
)


def thin_curve(
    x_values: np.ndarray, y_values: np.ndarray, max_points: int = 1_000
) -> tuple[np.ndarray, np.ndarray]:
    '''Retain evenly spaced plotting points while leaving metrics exact.'''
    if len(x_values) <= max_points:
        return x_values, y_values
    positions = np.unique(np.linspace(0, len(x_values) - 1, max_points, dtype=int))
    return x_values[positions], y_values[positions]


curve_inputs = [
    ("Validation selection model", y_validation, validation_probability, "#2563EB"),
    ("Final test model", y_test, test_probability, "#D97706"),
]
for label, y_true, probability, color in curve_inputs:
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, probability)
    false_positive_rate, true_positive_rate = thin_curve(
        false_positive_rate, true_positive_rate
    )
    diagnostic_figure.add_trace(
        go.Scatter(
            x=false_positive_rate,
            y=true_positive_rate,
            mode="lines",
            name=f"{label} (AUC={roc_auc_score(y_true, probability):.3f})",
            line={"color": color},
            legendgroup=label,
        ),
        row=1,
        col=1,
    )

    precision, recall, _ = precision_recall_curve(y_true, probability)
    recall, precision = thin_curve(recall, precision)
    diagnostic_figure.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=f"{label} (AP={average_precision_score(y_true, probability):.3f})",
            line={"color": color},
            legendgroup=label,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    observed_rate, predicted_rate = calibration_curve(
        y_true, probability, n_bins=10, strategy="quantile"
    )
    diagnostic_figure.add_trace(
        go.Scatter(
            x=predicted_rate,
            y=observed_rate,
            mode="lines+markers",
            name=label,
            line={"color": color},
            legendgroup=label,
            showlegend=False,
        ),
        row=2,
        col=1,
    )

for row_number, column_number in [(1, 1), (2, 1)]:
    diagnostic_figure.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            line={"color": "#6B7280", "dash": "dash"},
            showlegend=False,
        ),
        row=row_number,
        col=column_number,
    )

for outcome, label, color in [
    (0, "Actually incorrect", "#DC2626"),
    (1, "Actually correct", "#059669"),
]:
    histogram_counts, histogram_edges = np.histogram(
        test_probability[y_test.to_numpy() == outcome], bins=np.linspace(0, 1, 31)
    )
    diagnostic_figure.add_trace(
        go.Bar(
            x=(histogram_edges[:-1] + histogram_edges[1:]) / 2,
            y=histogram_counts / histogram_counts.sum(),
            width=np.diff(histogram_edges),
            opacity=0.55,
            name=label,
            marker_color=color,
        ),
        row=2,
        col=2,
    )

diagnostic_figure.update_xaxes(title_text="False positive rate", row=1, col=1)
diagnostic_figure.update_yaxes(title_text="True positive rate", row=1, col=1)
diagnostic_figure.update_xaxes(title_text="Recall", row=1, col=2)
diagnostic_figure.update_yaxes(title_text="Precision", row=1, col=2)
diagnostic_figure.update_xaxes(title_text="Mean predicted probability", row=2, col=1)
diagnostic_figure.update_yaxes(title_text="Observed correct rate", row=2, col=1)
diagnostic_figure.update_xaxes(title_text="Predicted probability of correct", row=2, col=2)
diagnostic_figure.update_yaxes(title_text="Share within outcome", row=2, col=2)
diagnostic_figure.update_layout(
    title="XGBoost performance diagnostics",
    template="plotly_white",
    barmode="overlay",
    height=850,
    legend={"orientation": "h", "y": -0.12},
)
diagnostic_figure.show()


## Test confusion matrix at the validation-selected threshold


In [10]:
test_prediction = (test_probability >= SELECTED_THRESHOLD).astype("int8")
test_confusion = confusion_matrix(y_test, test_prediction, labels=[0, 1])
confusion_table = pd.DataFrame(
    test_confusion,
    index=["Actual incorrect", "Actual correct"],
    columns=["Predicted incorrect", "Predicted correct"],
)
display(confusion_table)

confusion_figure = go.Figure(
    data=go.Heatmap(
        z=test_confusion,
        x=confusion_table.columns,
        y=confusion_table.index,
        text=test_confusion,
        texttemplate="%{text:,}",
        colorscale="Blues",
        showscale=False,
    )
)
confusion_figure.update_layout(
    title=f"Test confusion matrix (threshold = {SELECTED_THRESHOLD:.3f})",
    template="plotly_white",
    width=700,
    height=500,
    xaxis_title="Model classification",
    yaxis_title="Observed outcome",
)
confusion_figure.show()


,Predicted incorrect,Predicted correct
Actual incorrect,5292,5724
Actual correct,6050,21843


## Global explainability: gain and native SHAP contributions

Gain measures how much splits using a feature improved the tree objective. Native SHAP
contributions provide an additive log-odds decomposition and are averaged over a
deterministic test sample for a more comparable global ranking. Correlated features can
share or exchange importance, so neither ranking is causal.


In [11]:
dictionary_lookup = feature_dictionary.set_index("column_name")


def readable_feature_label(feature_name: str) -> str:
    '''Attach a plain-language skill label when the dictionary provides one.'''
    plain_name = dictionary_lookup.at[feature_name, "skill_plain_language_name"]
    if feature_name.startswith("Skill_") and plain_name:
        return f"{feature_name}: {plain_name}"
    return feature_name


booster = final_model.get_booster()
gain_scores = booster.get_score(importance_type="gain")
gain_table = pd.DataFrame(
    {
        "feature": feature_names,
        "feature_label": [readable_feature_label(name) for name in feature_names],
        "gain": [gain_scores.get(name, 0.0) for name in feature_names],
    }
).sort_values("gain", ascending=False, ignore_index=True)
gain_table["gain_share"] = gain_table["gain"] / gain_table["gain"].sum()

SHAP_SAMPLE_SIZE = min(10_000, len(test))
rng = np.random.default_rng(RANDOM_STATE)
shap_positions = np.sort(
    rng.choice(len(test), size=SHAP_SAMPLE_SIZE, replace=False)
)
shap_sample = test.iloc[shap_positions][feature_names]
shap_matrix = booster.predict(
    xgb.DMatrix(shap_sample, feature_names=feature_names),
    pred_contribs=True,
)
assert shap_matrix.shape == (SHAP_SAMPLE_SIZE, len(feature_names) + 1)

shap_importance_table = pd.DataFrame(
    {
        "feature": feature_names,
        "feature_label": [readable_feature_label(name) for name in feature_names],
        "mean_absolute_shap": np.abs(shap_matrix[:, :-1]).mean(axis=0),
        "mean_shap": shap_matrix[:, :-1].mean(axis=0),
    }
).sort_values("mean_absolute_shap", ascending=False, ignore_index=True)

display(gain_table.head(25))
display(shap_importance_table.head(25))


,feature,feature_label,gain,gain_share
0,mean_skill_recent_accuracy_5,mean_skill_recent_accuracy_5,823.8165,0.2115
1,min_skill_recent_accuracy_3,min_skill_recent_accuracy_3,224.0912,0.0575
2,mean_skill_prior_accuracy,mean_skill_prior_accuracy,167.3387,0.0430
3,student_incorrect_streak,student_incorrect_streak,156.8676,0.0403
4,mean_skill_recent_accuracy_3,mean_skill_recent_accuracy_3,129.2384,0.0332
5,mean_skill_recent_accuracy_10,mean_skill_recent_accuracy_10,127.8907,0.0328
6,Skill_2,Skill_2: Circle Graph,44.2306,0.0114
7,min_skill_recent_accuracy_5,min_skill_recent_accuracy_5,42.1487,0.0108
8,Skill_14,Skill_14: Mode,39.9814,0.0103
9,Skill_47,Skill_47: Conversion of Fraction Decimals Percents,38.4958,0.0099


,feature,feature_label,mean_absolute_shap,mean_shap
0,student_prior_accuracy,student_prior_accuracy,0.1472,0.0594
1,min_skill_prior_attempts,min_skill_prior_attempts,0.1311,0.0243
2,min_skill_recent_accuracy_3,min_skill_recent_accuracy_3,0.1302,0.0395
3,mean_skill_recent_accuracy_5,mean_skill_recent_accuracy_5,0.1217,0.0419
4,student_incorrect_streak,student_incorrect_streak,0.1159,0.0642
5,hint_total,hint_total,0.1068,0.0206
6,student_correct_streak,student_correct_streak,0.0917,0.0092
7,mean_skill_prior_accuracy,mean_skill_prior_accuracy,0.0898,0.0228
8,position,position,0.0809,0.0632
9,student_prior_correct,student_prior_correct,0.0764,0.0607


In [12]:
shap_plot_data = shap_importance_table.head(25).sort_values("mean_absolute_shap")
shap_figure = go.Figure(
    go.Bar(
        x=shap_plot_data["mean_absolute_shap"],
        y=shap_plot_data["feature_label"],
        orientation="h",
        marker_color="#7C3AED",
        hovertemplate=(
            "%{y}<br>Mean |SHAP|: %{x:.4f} log-odds<extra></extra>"
        ),
    )
)
shap_figure.update_layout(
    title=f"Global native SHAP importance ({SHAP_SAMPLE_SIZE:,} test rows)",
    template="plotly_white",
    height=800,
    xaxis_title="Mean absolute SHAP contribution (log-odds)",
    yaxis_title=None,
    margin={"l": 330},
)
shap_figure.show()


## Local native SHAP explanation for one retrospective test prediction


In [13]:
representative_position = int(np.argmin(np.abs(test_probability - np.median(test_probability))))
representative_row = test.iloc[[representative_position]]
representative_probability = float(test_probability[representative_position])
local_shap = booster.predict(
    xgb.DMatrix(representative_row[feature_names], feature_names=feature_names),
    pred_contribs=True,
)[0]
reconstructed_probability = float(expit(local_shap.sum()))
assert np.isclose(representative_probability, reconstructed_probability, atol=1e-6)

local_explanation = pd.DataFrame(
    {
        "feature": feature_names,
        "feature_label": [readable_feature_label(name) for name in feature_names],
        "raw_value": representative_row[feature_names].to_numpy().ravel(),
        "shap_log_odds_contribution": local_shap[:-1],
        "odds_multiplier": np.exp(local_shap[:-1]),
    }
)
local_explanation["absolute_contribution"] = local_explanation[
    "shap_log_odds_contribution"
].abs()
local_explanation = local_explanation.sort_values(
    "absolute_contribution", ascending=False, ignore_index=True
)

local_summary = pd.DataFrame(
    {
        "user_id": representative_row["user_id"].to_numpy(),
        "order_id": representative_row["order_id"].to_numpy(),
        "observed_correct": representative_row[TARGET].to_numpy(),
        "predicted_probability_correct": [representative_probability],
        "model_classification": [
            int(representative_probability >= SELECTED_THRESHOLD)
        ],
        "validation_selected_threshold": [SELECTED_THRESHOLD],
        "shap_bias_log_odds": [float(local_shap[-1])],
    }
)
display(local_summary)
display(local_explanation.head(15))


,user_id,order_id,observed_correct,predicted_probability_correct,model_classification,validation_selected_threshold,shap_bias_log_odds
0,96248,38289406,1,0.7417,1,0.6350,0.6123


,feature,feature_label,raw_value,shap_log_odds_contribution,odds_multiplier,absolute_contribution
0,Skill_1,Skill_1: Box and Whisker,1.0000,0.4231,1.5267,0.4231
1,student_correct_streak,student_correct_streak,0.0000,-0.2062,0.8137,0.2062
2,min_skill_recent_accuracy_3,min_skill_recent_accuracy_3,0.6667,0.1212,1.1289,0.1212
3,min_skill_prior_attempts,min_skill_prior_attempts,15.0000,0.1196,1.1270,0.1196
4,max_skill_recent_accuracy_3,max_skill_recent_accuracy_3,1.0000,-0.0990,0.9057,0.0990
5,student_prior_accuracy,student_prior_accuracy,0.7481,0.0976,1.1025,0.0976
6,mean_skill_recent_accuracy_5,mean_skill_recent_accuracy_5,0.7000,0.0883,1.0923,0.0883
7,Skill_13,Skill_13: Median,1.0000,-0.0768,0.9261,0.0768
8,student_prior_hint_rate,student_prior_hint_rate,0.0489,0.0654,1.0676,0.0654
9,median_skill_recent_accuracy_3,median_skill_recent_accuracy_3,0.8333,-0.0608,0.9410,0.0608


## Interpretation and next steps

Review XGBoost against the logistic baseline primarily on held-out log loss, Brier score,
ROC AUC, and average precision. A nonlinear model is useful only if its test improvement
is meaningful, its calibration is acceptable, and its behavior remains stable across
time and student groups.

Before any teacher-facing use, investigate the validation-to-test population change,
perform subgroup and temporal stability audits, select an intervention threshold using
educational costs, and review explanations with educators. Predictions must not
automatically determine grades, placement, or interventions and are not measures of
intelligence, motivation, disability, or general ability.


## Export deployable XGBoost artifact

The final classifier is saved in XGBoost's native JSON format for portable serving. A
separate metadata contract records the ordered predictors, validation-selected threshold,
held-out metrics, package versions, and reference predictions used for deployment checks.
Teacher Support Studio loads this artifact rather than retraining at application startup.

In [14]:
import json
from datetime import datetime, timezone

ARTIFACT_DIR = PROJECT_ROOT / "models" / "xgboost"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ARTIFACT_PATH = ARTIFACT_DIR / "xgboost_first_attempt.json"
MODEL_METADATA_PATH = ARTIFACT_DIR / "xgboost_first_attempt_metadata.json"

# XGBoost's native JSON format is language-portable and avoids pickle compatibility risk.
final_model.save_model(MODEL_ARTIFACT_PATH)

selected_test_metrics = test_performance.iloc[1].to_dict()
metadata = {
    "artifact_version": 1,
    "model_family": "xgboost",
    "model_format": "xgboost_native_json",
    "source_notebook": "notebooks/05_xgboost_model.ipynb",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "target": TARGET,
    "positive_class": 1,
    "prediction_semantics": "Probability of a correct first attempt for one interaction.",
    "feature_names": feature_names,
    "feature_count": len(feature_names),
    "selected_threshold": SELECTED_THRESHOLD,
    "training_scope": "chronological train and validation splits",
    "selection": {
        "configuration": BEST_CONFIGURATION_NAME,
        "boosting_rounds": BEST_BOOSTING_ROUNDS,
        "parameters": BEST_PARAMETERS,
    },
    "test_metrics": selected_test_metrics,
    "library_versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "xgboost": xgb.__version__,
    },
    "reference_predictions": [
        {
            "order_id": int(test.iloc[position]["order_id"]),
            "probability": float(test_probability[position]),
        }
        for position in range(min(5, len(test)))
    ],
}
MODEL_METADATA_PATH.write_text(
    json.dumps(metadata, indent=2, default=lambda value: value.item()),
    encoding="utf-8",
)

loaded_model = XGBClassifier()
loaded_model.load_model(MODEL_ARTIFACT_PATH)
loaded_probability = loaded_model.predict_proba(X_test.iloc[:5])[:, 1]
np.testing.assert_allclose(loaded_probability, test_probability[:5], rtol=0, atol=1e-7)

print(f"Saved deployable model: {MODEL_ARTIFACT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved metadata contract: {MODEL_METADATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Artifact size: {MODEL_ARTIFACT_PATH.stat().st_size / 1024**2:.2f} MiB")


Saved deployable model: models\xgboost\xgboost_first_attempt.json
Saved metadata contract: models\xgboost\xgboost_first_attempt_metadata.json
Artifact size: 2.64 MiB
